In [ ]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()

model = init_chat_model(
    "google_genai:gemini-2.5-flash-lite",
    temperature=0
)

In [2]:
from typing import TypedDict

class TranslationState(TypedDict):
    original: str
    translated: str
    review: str
    final_output: str

In [3]:
def translate(state: TranslationState) -> dict:
    """1단계: 영어 → 한국어 번역"""
    msg = model.invoke(
        f"다음 영어 문장을 한국어로 번역하세요. 번역문만 출력하세요.\n\n{state['original']}"
    )
    print(f"[translate] {msg.content}")
    return {"translated": msg.content}

def review(state: TranslationState) -> dict:
    """2단계: 번역 품질 검수"""
    msg = model.invoke(
        f"다음 번역의 품질을 한 줄로 평가하세요.\n"
        f"원문: {state['original']}\n번역: {state['translated']}"
    )
    print(f"[review] {msg.content}")
    return {"review": msg.content}

def finalize(state: TranslationState) -> dict:
    """3단계: 최종 출력 생성"""
    final = f"번역: {state['translated']}\n검수: {state['review']}"
    print(f"[finalize] 완료")
    return {"final_output": final}

In [4]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

# 그래프 구성
builder = StateGraph(TranslationState)

builder.add_node("translate", translate)
builder.add_node("review", review)
builder.add_node("finalize", finalize)

builder.add_edge(START, "translate")
builder.add_edge("translate", "review")
builder.add_edge("review", "finalize")
builder.add_edge("finalize", END)

# 체크포인터와 함께 컴파일
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [5]:
config = {"configurable": {"thread_id": "translation_1"}}

result = graph.invoke(
    {
        "original": "The early bird catches the worm.",
        "translated": "",
        "review": "",
        "final_output": "",
    },
    config=config,
)


Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


[translate] 일찍 일어나는 새가 벌레를 잡는다.


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


[review] 이 번역은 원문의 의미를 정확하고 자연스럽게 전달하는 훌륭한 번역입니다.
[finalize] 완료


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [8]:
result

{'original': 'The early bird catches the worm.',
 'translated': '일찍 일어나는 새가 벌레를 잡는다.',
 'review': '이 번역은 원문의 의미를 정확하고 자연스럽게 전달하는 훌륭한 번역입니다.',
 'final_output': '번역: 일찍 일어나는 새가 벌레를 잡는다.\n검수: 이 번역은 원문의 의미를 정확하고 자연스럽게 전달하는 훌륭한 번역입니다.'}

In [9]:
graph.get_state_history(config)

<generator object Pregel.get_state_history at 0x0000023762FF10C0>

In [10]:
history = list(graph.get_state_history(config))

In [11]:
history

[StateSnapshot(values={'original': 'The early bird catches the worm.', 'translated': '일찍 일어나는 새가 벌레를 잡는다.', 'review': '이 번역은 원문의 의미를 정확하고 자연스럽게 전달하는 훌륭한 번역입니다.', 'final_output': '번역: 일찍 일어나는 새가 벌레를 잡는다.\n검수: 이 번역은 원문의 의미를 정확하고 자연스럽게 전달하는 훌륭한 번역입니다.'}, next=(), config={'configurable': {'thread_id': 'translation_1', 'checkpoint_ns': '', 'checkpoint_id': '1f10e10f-4655-664b-8003-0c7342fc635c'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-02-20T04:02:20.774356+00:00', parent_config={'configurable': {'thread_id': 'translation_1', 'checkpoint_ns': '', 'checkpoint_id': '1f10e10f-4651-6855-8002-b37f265d0ab6'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'original': 'The early bird catches the worm.', 'translated': '일찍 일어나는 새가 벌레를 잡는다.', 'review': '이 번역은 원문의 의미를 정확하고 자연스럽게 전달하는 훌륭한 번역입니다.', 'final_output': ''}, next=('finalize',), config={'configurable': {'thread_id': 'translation_1', 'checkpoint_ns': '', 'checkpoint_id': '1f10e10f-4651-6855-8002-b37f265d0ab6'

In [12]:
len(history)

5

In [13]:
for i, snapshot in enumerate(history):
    step = snapshot.metadata.get("step", "N/A")
    next_nodes = snapshot.next
    cp_id = snapshot.config["configurable"]["checkpoint_id"]

    print(f"[체크포인트 {i}] Step {step}")
    print(f"  다음 노드: {next_nodes}")
    print(f"  ID: {cp_id[:20]}...")

    # 상태 요약
    vals = snapshot.values
    if vals.get("translated"):
        print(f"  번역: {vals['translated'][:50]}...")
    print()

[체크포인트 0] Step 3
  다음 노드: ()
  ID: 1f10e10f-4655-664b-8...
  번역: 일찍 일어나는 새가 벌레를 잡는다....

[체크포인트 1] Step 2
  다음 노드: ('finalize',)
  ID: 1f10e10f-4651-6855-8...
  번역: 일찍 일어나는 새가 벌레를 잡는다....

[체크포인트 2] Step 1
  다음 노드: ('review',)
  ID: 1f10e10f-3f14-6b22-8...
  번역: 일찍 일어나는 새가 벌레를 잡는다....

[체크포인트 3] Step 0
  다음 노드: ('translate',)
  ID: 1f10e10f-3498-6793-8...

[체크포인트 4] Step -1
  다음 노드: ('__start__',)
  ID: 1f10e10f-3495-6eb5-b...



번역 수정 후 재실행

In [14]:
# review 실행 직전 체크포인트 찾기 (translate 완료 직후)
target = None
for snapshot in history:
    if snapshot.next == ("review",):
        target = snapshot
        break

print(f"선택한 체크포인트: Step {target.metadata.get('step')}")
print(f"현재 번역: {target.values['translated']}")

선택한 체크포인트: Step 1
현재 번역: 일찍 일어나는 새가 벌레를 잡는다.


In [17]:
target

StateSnapshot(values={'original': 'The early bird catches the worm.', 'translated': '일찍 일어나는 새가 벌레를 잡는다.', 'review': '', 'final_output': ''}, next=('review',), config={'configurable': {'thread_id': 'translation_1', 'checkpoint_ns': '', 'checkpoint_id': '1f10e10f-3f14-6b22-8001-7b8e99dcfad3'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-02-20T04:02:20.013853+00:00', parent_config={'configurable': {'thread_id': 'translation_1', 'checkpoint_ns': '', 'checkpoint_id': '1f10e10f-3498-6793-8000-ea92091708bc'}}, tasks=(PregelTask(id='20c043a0-9593-2bbc-518e-c9534ca3cd97', name='review', path=('__pregel_pull', 'review'), error=None, interrupts=(), state=None, result={'review': '이 번역은 원문의 의미를 정확하고 자연스럽게 전달하는 훌륭한 번역입니다.'}),), interrupts=())

In [15]:
# 번역을 의역으로 수정
graph.update_state(
    target.config,
    {"translated": "부지런한 자가 성공한다."},
    as_node="translate",  # translate가 업데이트한 것처럼 → 다음은 review
)

print("번역 수정 완료: '부지런한 자가 성공한다.'")

번역 수정 완료: '부지런한 자가 성공한다.'


In [ ]:
# None 입력으로 재실행 → 새로운 fork 생성
result2 = graph.invoke(None, config=target.config)

print("\n=== Time Travel 결과 ===")
print(result2["final_output"])
